# 第 10 周 · 周三：案例——新功能上线，到底有没有效果？

> 难度：⭐⭐⭐⭐⭐ | 定位：掌握无AB实验时的因果推断方法，区分"相关性"和"因果性"
> 前置依赖：第3周AB实验、第7周因果推断、第9周准实验方法
> 面试地位：高级数据分析岗位的核心区分点——能做好AB实验的人很多，能在不能做实验时依然给出可信结论的人很少
> 配对课程：01python_learn W10D3（不平衡数据处理）

---

## 📘 区块1：核心概念与第一性原理

### 1.1 为什么"不能做AB实验"是常态？

理想状态下，评估任何策略/功能变化都应该用随机对照实验（RCT）。但现实中，至少有50%的情况你无法做AB实验：

- **全量发布**：产品/运营执意全量上线，"灰度就是浪费时间"
- **无法随机**：策略本身无法随机分配（如"所有VIP用户自动获得新功能"）
- **干扰太大**：实验组和对照组会互相干扰（如社交功能、双边市场）
- **时间不够**：需要3个月才能看到效果，但决策等不了3个月
- **样本太小**：DAU才几千，分流后每组几百人，检验力不够

**关键认知**：不能做AB实验不代表不能做因果推断——只是需要更聪明的方法。

### 1.2 朴素对比为什么不可信？

产品经理最常说的逻辑是："上线后人均观看时长涨了12%，肯定有效！"

这个说法的致命缺陷是将"相关性"等同于"因果性"。上线后的人均观看时长由三部分组成：
```
观测值 = 自然趋势 + 季节效应 + 真实效果 + 随机噪声
```

如果"自然趋势"本身就让时长在涨（比如因为老用户越来越沉浸），那"上线后涨了12%"中可能有10%是趋势贡献的，策略只贡献了2%。

### 1.3 DID（Difference-in-Differences）的核心思想

DID是经济学家最常用的因果推断方法，它的核心思想可以用一句话概括：**"用对照组的变化来近似实验组如果没有策略会怎样"**。

```
处理效应 = (实验组后 - 实验组前) - (对照组后 - 对照组前)
         = 实验组前后变化 - 对照组前后变化
```

DID的关键假设是"平行趋势"：在没有策略的情况下，实验组和对照组的变化趋势应该是一样的。如果这个假设不成立，DID的估计就是有偏的。

### 1.4 当没有对照组时——"准DID"

如果全量上线导致没有对照组，可以用"趋势外推"替代对照组：
1. 用上线前的历史数据拟合一条趋势线
2. 将趋势线外推到上线后，作为"如果不上线"的预测值
3. 实际值 - 预测值 = 处理效应的近似估计

这种方法的致命弱点是：你假设"趋势会按原来的方式继续"——但实际上趋势可能会弯折、加速、减速。所以准DID的结论需要加"置信区间"，而且结论中必须写明"基于趋势外推的假设"。

### 1.5 评估新功能效果的五个维度

**为什么重要**：只看"核心指标涨没涨"太单薄了，需要多维度交叉验证才能确信效果是真实的。

| 维度 | 核心问题 | 分析方法 |
|------|---------|--------|
| 直接效果 | 核心指标变了吗？ | DID/ITS/Causal Impact |
| 异质性 | 谁受益最大？谁最差？ | 分组DID、CATE（条件平均处理效应） |
| 溢出效应 | 有没有负面影响？ | 检查不受影响的相关指标 |
| 持续性 | 效果在衰减还是在放大？ | 分时间段看效应的变化趋势 |
| 机制验证 | 为什么有效（或无效）？ | 中介分析——检查中间行为指标 |

### 1.6 面试高频考点

**面试官常问**：
- "如果没有AB实验，你怎么评估功能效果？"
- "DID的平行趋势假设是什么？如何检验？"
- "PM说上线后指标涨了XX%，你怎么判断是不是真的有效？"
- "什么情况下朴素对比反而比DID更可靠？"

**回答要点**：
- 没有实验时使用DID、ITS、Causal Impact、PSM等方法
- 平行趋势检验：看上线的处理组和对照组在干预前的趋势是否平行
- 剥离自然趋势：用上线前数据拟合趋势线，外推后对比
- 当数据非常稳定、几乎没有趋势和季节性时，朴素对比也可以接受

## 🧩 区块2：知识点详细拆解

### 2.1 DID的数学原理与假设

**为什么重要**：DID是无实验条件下最重要的因果推断方法，理解其原理是正确使用的前提。

**模型设定**：
```
Y_it = β₀ + β₁·Treated_i + β₂·Post_t + β₃·(Treated_i × Post_t) + ε_it

其中：
- Treated_i = 1 表示个体i在处理组
- Post_t = 1 表示时间t在干预后
- β₃ = DID估计量 = 平均处理效应(ATT)
```

**关键假设**：
1. **平行趋势假设**：无干预时，两组的变化趋势相同（最重要的假设）
2. **共同冲击假设**：两组受到相同的外部冲击
3. **无溢出效应**：处理组的干预不影响对照组（SUTVA）
4. **组合不变性**：处理组和对照组的组成在干预前后不变

### 2.2 平行趋势检验——DID的生命线

**为什么重要**：如果平行趋势不成立，整个DID分析就是废纸。

**检验方法**：

| 方法 | 操作 | 判断标准 |
|------|------|--------|
| 可视化检查 | 画两组干预前的时间序列，看是否大致平行 | 两条线不能交叉或发散 |
| 事件研究法 | 对每个干预前时间点做DID回归，看系数是否为零 | 所有干预前系数不显著≠0 |
| 安慰剂检验 | 把干预时间"假装"提前到实际干预前，看是否有"效果" | 虚假干预时间的DID系数不显著 |
| 趋势差异检验 | 回归中加组别×时间趋势的交互项 | 交互项系数不显著 |

**当平行趋势不成立时怎么办**：
- 使用合成控制法（SCM）——可以处理趋势不平行的情况
- 使用断点回归（RDD）——如果有明确的断点
- 加控制变量——如果趋势差异可以用控制变量解释

### 2.3 时间序列中断分析（ITS）

**为什么重要**：当只有一个处理组且没有对照组时，ITS是DID的最佳替代方案。

**ITS模型**：
```
Y_t = β₀ + β₁·Time_t + β₂·Post_t + β₃·Time_After_t + ε_t

其中：
- Time_t: 从0开始的时间变量
- Post_t: 干预后的指示变量（0/1）
- Time_After_t: 干预后的时间变量（干预前=0）
- β₂ = 水平变化（immediate effect）
- β₃ = 斜率变化（趋势变化，即长期效应）
```

**ITS的核心输出**：
- β₂：干预即刻带来的水平变化（"跳变"）
- β₃：干预后趋势的斜率变化（"加速/减速"）
- β₂ + β₃ × N：干预后第N期的总效应

### 2.4 合成控制法（SCM）简介

**为什么重要**：当我们有多个潜在的对照单元时，SCM可以构造一个"合成的反事实"。

**核心思想**：
1. 从"候选对照池"中选出一组单元
2. 用加权平均的方式构造一个"合成对照"
3. 权重通过最小化干预前的预测误差确定
4. 干预效应 = 处理组的实际值 - 合成对照的预测值

**SCM的优势**：
- 不依赖平行趋势假设
- 权重选择是数据驱动的（而非主观选取）
- 可以处理时变的混杂因素

**SCM的局限**：
- 需要足够长的干预前数据
- 需要足够多的候选对照单元
- 如果处理组在干预前就极端（比如最高的那个），合成不出来好的对照

### 2.5 Causal Impact（贝叶斯结构时间序列）

**为什么重要**：Google开发的Causal Impact方法可以给出"处理效应的后验分布"，比点估计更有信息量。

**工作原理**：
1. 用干预前的数据训练一个贝叶斯结构时间序列模型
2. 模型自动选择与处理组最相关的对照组变量
3. 外推预测"如果无干预"的路径
4. 输出：效应的后验分布（均值+置信区间）、效应的累计值、效应的时序变化

**Causal Impact vs DID**：

| 维度 | DID | Causal Impact |
|------|-----|--------------|
| 假设 | 平行趋势 | 对照组能预测处理组 |
| 输出 | 点估计 | 后验分布 |
| 适用场景 | 有清晰的处理/对照组划分 | 有多个时间序列作为对照 |
| 难度 | 低 | 中 |

### 2.6 安慰剂检验——因果推断的"测谎仪"

**为什么重要**：安慰剂检验是检验因果推断方法是否可靠的终极武器。

**两种安慰剂检验**：

**时间安慰剂（In-Time Placebo）**：
- 把干预时间"假装"提前到实际干预之前
- 对"虚假干预"运行同样的因果推断方法
- 如果发现"效果"——说明你的方法不可靠
- 如果没发现——说明方法可能在真实干预上是可靠的

**空间安慰剂（In-Space Placebo）**：
- 把"未受干预的单元"当作"假处理组"
- 对假处理组运行同样的因果推断方法
- 应该得到"无效果"的结论

### 2.7 效应异质性分析——"平均"掩盖了什么？

**为什么重要**：策略效果对不同人群可能完全不同，平均效应可能掩盖"对A人群极其有效，对B人群反而有害"的事实。

**分组DID（Staggered DID）**：
```
高活用户: DID = +5.2分钟（显著）
中活用户: DID = +1.8分钟（微弱显著）
低活用户: DID = -3.1分钟（显著为负！新功能反而让他们困惑）
```

**CATE（条件平均处理效应）**：用因果森林或因果树来估计不同用户特征下的处理效应，避免了手动分组的任意性。

### 2.8 为什么"不做实验"会高估效果？

**为什么重要**：超过60%的公司内部"成功案例"在严格评估后被发现高估了效果（有时效果甚至为零或为负）。

**高估的六大来源**：

| 高估来源 | 影响方向 | 例子 |
|---------|---------|------|
| 自然趋势 | 高估 | 用户本身就在变得更活跃 |
| 选择效应 | 高估 | 第一批使用新功能的是高活用户 |
| 霍桑效应 | 高估 | 用户因为"有变化"而暂时更活跃 |
| 新奇效应 | 高估 | 新功能新鲜感带来短期使用高峰 |
| 指标替代 | 高估 | 使用了新功能→减少了其他更重要的行为 |
| 回归均值 | 不确定 | 极端用户自然会向均值回归 |

**应对**：每一个"效果"报告都应该附上"可能高估的来源"和对应的"矫正方案"。

## 💻 区块3：动手练习

### 步骤0：环境与数据准备

生成完整的"新功能上线"数据，包含自然趋势、季节性、真实效应和噪声。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)

# 模拟数据设定
n_weeks_before = 16  # 上线前16周
n_weeks_after = 12   # 上线后12周
n_weeks = n_weeks_before + n_weeks_after
weeks = np.arange(-n_weeks_before, n_weeks_after)

# 真实的处理效应
TRUE_EFFECT = 5.0  # 新功能让观看时长增加5分钟

# 自然趋势（每周增长）
time_trend = 0.3  # 每周自然增长0.3分钟

# 季节性（周期=8周）
seasonality_amplitude = 3.0

# 噪声
noise_std = 2.0

def generate_duration_data(with_effect=True):
    """生成周度人均观看时长数据"""
    trend = time_trend * weeks
    seasonality = seasonality_amplitude * np.sin(2 * np.pi * weeks / 8)
    effect = TRUE_EFFECT * (weeks >= 0) if with_effect else 0
    noise = np.random.normal(0, noise_std, n_weeks)
    
    duration = 60 + trend + seasonality + effect + noise
    
    df = pd.DataFrame({
        'week': weeks,
        'period': ['before' if w < 0 else 'after' for w in weeks],
        'duration': np.round(duration, 1),
        'trend': 60 + trend,
        'seasonality': seasonality,
        'effect': effect,
        'noise': noise
    })
    return df

df = generate_duration_data(with_effect=True)

print("数据生成完成")
print(f"干预前均值: {df[df['period']=='before']['duration'].mean():.1f}分钟")
print(f"干预后均值: {df[df['period']=='after']['duration'].mean():.1f}分钟")
print(f"朴素对比: {df[df['period']=='after']['duration'].mean() - df[df['period']=='before']['duration'].mean():+.1f}分钟")
print(f"真实效应: {TRUE_EFFECT}分钟")

### 步骤1：基础练习——朴素对比 vs 趋势剥离

1.1 朴素前后对比
1.2 简单趋势剥离
1.3 可视化对比

In [ ]:
# 步骤1.1：朴素前后对比——PM最喜欢的方法

before_mean = df[df['period'] == 'before']['duration'].mean()
after_mean = df[df['period'] == 'after']['duration'].mean()
naive_effect = after_mean - before_mean

print("=" * 50)
print("方法1: 朴素前后对比")
print("=" * 50)
print(f"上线前均值: {before_mean:.1f}分钟")
print(f"上线后均值: {after_mean:.1f}分钟")
print(f"估计效应: {naive_effect:+.1f}分钟 ({naive_effect/before_mean*100:+.1f}%)")
print(f"真实效应: {TRUE_EFFECT:+.1f}分钟")
print(f"偏差: {naive_effect - TRUE_EFFECT:+.1f}分钟 (高估了{(naive_effect-TRUE_EFFECT)/TRUE_EFFECT*100:.0f}%!)")
print(f"\n原因: 朴素对比把自然趋势({time_trend*n_weeks_after:.1f}分钟增长)也算成了功能效果")

In [ ]:
# 步骤1.2：简单趋势剥离——用上线前的线性趋势预测"如果没有新功能会怎样"

def simple_trend_detrend(df):
    """
    用上线前数据拟合线性趋势，外推后剥离
    """
    before = df[df['period'] == 'before'].copy()
    after = df[df['period'] == 'after'].copy()
    
    # 用上线前数据拟合线性趋势
    X_before = before['week'].values.reshape(-1, 1)
    y_before = before['duration'].values
    model = LinearRegression()
    model.fit(X_before, y_before)
    
    slope = model.coef_[0]
    intercept = model.intercept_
    
    # 预测上线后各周（如果无干预）
    X_after = after['week'].values.reshape(-1, 1)
    predicted_no_treatment = model.predict(X_after)
    
    # 实际值 - 预测值 = 趋势剥离后的效应
    actual = after['duration'].values
    detrended_effect = actual - predicted_no_treatment
    
    avg_effect = detrended_effect.mean()
    
    print("=" * 50)
    print("方法2: 线性趋势剥离")
    print("=" * 50)
    print(f"拟合的趋势斜率: {slope:.3f}分钟/周")
    print(f"估计自然增长率: {slope*n_weeks_after:.1f}分钟（16周累计）")
    print(f"\n上线后实际均值: {actual.mean():.1f}分钟")
    print(f"趋势预测(无新功能): {predicted_no_treatment.mean():.1f}分钟")
    print(f"剥离趋势后效应: {avg_effect:+.1f}分钟")
    print(f"真实效应: {TRUE_EFFECT:+.1f}分钟")
    print(f"偏差: {avg_effect - TRUE_EFFECT:+.1f}分钟")
    
    return {
        'slope': slope,
        'predicted': predicted_no_treatment,
        'actual': actual,
        'effect': avg_effect
    }

detrend_result = simple_trend_detrend(df)

In [ ]:
# 步骤1.3：三种方法的可视化对比

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 图1: 朴素对比
ax1 = axes[0]
ax1.plot(df['week'], df['duration'], 'b-o', linewidth=1.5, markersize=4, label='实际值')
ax1.axvline(x=-0.5, color='r', linestyle='--', linewidth=1.5, label='上线时间')
ax1.axhline(y=before_mean, xmin=0, xmax=0.57, color='gray', linestyle='-', alpha=0.5)
ax1.axhline(y=after_mean, xmin=0.57, xmax=1, color='gray', linestyle='-', alpha=0.5)
ax1.fill_between(df['week'], before_mean, after_mean, alpha=0.1, color='green', label=f'朴素效应={naive_effect:.1f}')
ax1.set_title('方法1: 朴素前后对比\n(高估!)', fontsize=13)
ax1.set_xlabel('周（0=上线周）')
ax1.set_ylabel('人均观看时长(分钟)')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# 图2: 趋势剥离
ax2 = axes[1]
ax2.plot(df['week'], df['duration'], 'b-o', linewidth=1.5, markersize=4, label='实际值')
ax2.axvline(x=-0.5, color='r', linestyle='--', linewidth=1.5, label='上线时间')

# 延伸的趋势线
all_weeks = df['week'].values.reshape(-1, 1)
before_mask = df['week'] < 0
lr = LinearRegression()
lr.fit(df.loc[before_mask, 'week'].values.reshape(-1, 1), df.loc[before_mask, 'duration'].values)
trend_line = lr.predict(all_weeks)
ax2.plot(df['week'], trend_line, 'g--', linewidth=1.5, alpha=0.7, label='趋势外推(无干预)')

# 趋势剥离后的效应
after_mask = df['week'] >= 0
ax2.fill_between(df.loc[after_mask, 'week'],
                trend_line[after_mask],
                df.loc[after_mask, 'duration'].values,
                alpha=0.2, color='green', label=f'剥离后效应={detrend_result["effect"]:.1f}')
ax2.set_title('方法2: 线性趋势剥离\n(较准确)', fontsize=13)
ax2.set_xlabel('周（0=上线周）')
ax2.set_ylabel('人均观看时长(分钟)')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# 图3: 完整DID思路
ax3 = axes[1]
ax3.plot(df['week'], df['duration'], 'b-o', linewidth=1.5, markersize=4, label='处理组(实际)')
ax3.axvline(x=-0.5, color='r', linestyle='--', linewidth=1.5, label='上线时间')

# 模拟对照组（无新功能）
df_control = generate_duration_data(with_effect=False)
ax3.plot(df_control['week'], df_control['duration'], 'orange', marker='s',
         linewidth=1.5, markersize=4, alpha=0.7, label='对照组(假设)')

# DID = (处理后-处理前) - (对照后-对照前)
treated_before = df[df['period']=='before']['duration'].mean()
treated_after = df[df['period']=='after']['duration'].mean()
control_before = df_control[df_control['period']=='before']['duration'].mean()
control_after = df_control[df_control['period']=='after']['duration'].mean()
did_effect = (treated_after - treated_before) - (control_after - control_before)

ax3.set_title(f'方法3: DID思路\nDID估计={did_effect:.1f}分钟', fontsize=13)
ax3.set_xlabel('周（0=上线周）')
ax3.set_ylabel('人均观看时长(分钟)')
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.3)

plt.suptitle('新功能效果评估：三种方法对比', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n总结:")
print(f"  朴素对比估计: {naive_effect:+.1f}分钟 (偏差: {naive_effect-TRUE_EFFECT:+.1f})")
print(f"  趋势剥离估计: {detrend_result['effect']:+.1f}分钟 (偏差: {detrend_result['effect']-TRUE_EFFECT:+.1f})")
print(f"  DID估计: {did_effect:+.1f}分钟 (偏差: {did_effect-TRUE_EFFECT:+.1f})")

### 步骤2：进阶练习——DID完整实现

2.1 标准DID回归
2.2 平行趋势检验
2.3 事件研究法

In [ ]:
# 步骤2.1：完整DID实现

def did_analysis(df_treated, df_control, intervention_week=0):
    """
    标准DID分析——面板数据结构
    """
    # 构造面板数据
    df_treated = df_treated.copy()
    df_treated['treated'] = 1
    df_control = df_control.copy()
    df_control['treated'] = 0
    
    df_panel = pd.concat([df_treated, df_control], ignore_index=True)
    df_panel['post'] = (df_panel['week'] >= intervention_week).astype(int)
    df_panel['did'] = df_panel['treated'] * df_panel['post']
    
    # DID回归: duration ~ treated + post + treated*post
    X = df_panel[['treated', 'post', 'did']].values
    X = np.column_stack([np.ones(len(X)), X])  # 加截距
    y = df_panel['duration'].values
    
    beta = np.linalg.inv(X.T @ X) @ X.T @ y
    
    # 简单的标准误计算
    y_pred = X @ beta
    residuals = y - y_pred
    n, k = X.shape
    sigma2 = np.sum(residuals**2) / (n - k)
    se = np.sqrt(np.diag(sigma2 * np.linalg.inv(X.T @ X)))
    t_stats = beta / se
    
    print("=" * 55)
    print("DID回归结果")
    print("=" * 55)
    print(f"模型: duration ~ β₀ + β₁·Treated + β₂·Post + β₃·Treated×Post")
    print(f"\n{'变量':<15}{'系数':>10}{'标准误':>10}{'t值':>10}{'解读':>20}")
    print("-" * 55)
    
    var_names = ['截距(β₀)', '组别(β₁)', '时间(β₂)', 'DID(β₃)']
    interpretations = [
        '对照组干预前水平',
        '处理组vs对照组的平均差异',
        '时间推移对所有组的影响',
        f'★ 处理效应（核心关注）'
    ]
    
    for i, (name, interp) in enumerate(zip(var_names, interpretations)):
        stars = ''
        if abs(t_stats[i]) > 2.58:
            stars = '***'
        elif abs(t_stats[i]) > 1.96:
            stars = '**'
        elif abs(t_stats[i]) > 1.64:
            stars = '*'
        print(f"{name:<15}{beta[i]:>10.3f}{se[i]:>10.3f}{t_stats[i]:>10.2f}{stars:>5}  {interp:<20}")
    
    did_effect = beta[3]
    print(f"\nDID估计的处理效应: {did_effect:+.2f} (p≈{2*(1-stats.norm.cdf(abs(t_stats[3]))):.3f})")
    print(f"真实效应: {TRUE_EFFECT:+.2f}")
    
    return {'did_estimate': did_effect, 'se': se[3], 't_stat': t_stats[3], 'beta': beta}

# 生成对照组数据
df_treated = generate_duration_data(with_effect=True)
df_control = generate_duration_data(with_effect=False)

did_result = did_analysis(df_treated, df_control)

In [ ]:
# 步骤2.2：平行趋势检验

def parallel_trend_test(df_treated, df_control, intervention_week=0):
    """
    检验干预前两组的趋势是否平行
    """
    before_treated = df_treated[df_treated['week'] < intervention_week]
    before_control = df_control[df_control['week'] < intervention_week]
    
    # 对每组拟合线性趋势
    X_t = before_treated['week'].values.reshape(-1, 1)
    y_t = before_treated['duration'].values
    lr_t = LinearRegression().fit(X_t, y_t)
    
    X_c = before_control['week'].values.reshape(-1, 1)
    y_c = before_control['duration'].values
    lr_c = LinearRegression().fit(X_c, y_c)
    
    slope_diff = lr_t.coef_[0] - lr_c.coef_[0]
    
    # 画图
    fig, ax = plt.subplots(figsize=(12, 5))
    
    ax.plot(before_treated['week'], before_treated['duration'], 'b-o', label='处理组(干预前)', linewidth=1.5, markersize=4)
    ax.plot(before_control['week'], before_control['duration'], 'orange', marker='s', label='对照组(干预前)', linewidth=1.5, markersize=4, alpha=0.7)
    
    # 趋势线
    x_range = np.linspace(before_treated['week'].min(), before_treated['week'].max(), 100).reshape(-1, 1)
    ax.plot(x_range, lr_t.predict(x_range), 'b--', alpha=0.5, label=f'处理组趋势(斜率={lr_t.coef_[0]:.3f})')
    ax.plot(x_range, lr_c.predict(x_range), 'orange', linestyle='--', alpha=0.5, label=f'对照组趋势(斜率={lr_c.coef_[0]:.3f})')
    
    ax.set_title(f'平行趋势检验 (斜率差异 = {slope_diff:.3f})', fontsize=14)
    ax.set_xlabel('周')
    ax.set_ylabel('人均观看时长(分钟)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n平行趋势检验结果:")
    print(f"  处理组干预前趋势斜率: {lr_t.coef_[0]:.4f}")
    print(f"  对照组干预前趋势斜率: {lr_c.coef_[0]:.4f}")
    print(f"  斜率差异: {slope_diff:.4f}")
    
    if abs(slope_diff) < 0.1:
        print(f"  ✓ 斜率差异极小，平行趋势假设基本成立")
    else:
        print(f"  ⚠ 斜率差异较大，平行趋势假设可能不成立——DID估计可能偏误")

parallel_trend_test(df_treated, df_control)

In [ ]:
# 步骤2.3：事件研究法——逐期DID估计

def event_study(df_treated, df_control, intervention_week=0, pre_periods=8, post_periods=8):
    """
    事件研究法：对每个时间点单独估计DID系数
    可以同时检验平行趋势和观察效应的时序变化
    """
    # 构造面板
    df_treated = df_treated.copy()
    df_treated['treated'] = 1
    df_control = df_control.copy()
    df_control['treated'] = 0
    df_panel = pd.concat([df_treated, df_control], ignore_index=True)
    
    # 为每个时期创建单独的DID项
    coefficients = []
    periods = []
    
    for t in range(-pre_periods, post_periods + 1):
        if t == -1:
            continue  # 基准期（干预前1期）
        
        period_mask = df_panel['week'] == t + intervention_week
        if period_mask.sum() == 0:
            continue
        
        # 简单计算：该时期的两组差异 - 基准期差异
        treated_val = df_panel[(df_panel['treated'] == 1) & period_mask]['duration'].mean()
        control_val = df_panel[(df_panel['treated'] == 0) & period_mask]['duration'].mean()
        
        # 基准期差异（t=-1）
        base_mask = df_panel['week'] == -1 + intervention_week
        treated_base = df_panel[(df_panel['treated'] == 1) & base_mask]['duration'].mean()
        control_base = df_panel[(df_panel['treated'] == 0) & base_mask]['duration'].mean()
        
        did_t = (treated_val - control_val) - (treated_base - control_base)
        
        coefficients.append(did_t)
        periods.append(t)
    
    # 画事件研究图
    fig, ax = plt.subplots(figsize=(12, 5))
    
    colors = ['gray' if p < 0 else 'blue' for p in periods]
    ax.bar(periods, coefficients, color=colors, alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.axvline(x=-0.5, color='r', linestyle='--', linewidth=1.5, label='干预时间')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.axhline(y=TRUE_EFFECT, color='green', linestyle='--', alpha=0.5, label=f'真实效应={TRUE_EFFECT}')
    
    ax.set_title('事件研究法: 逐期DID估计', fontsize=14, fontweight='bold')
    ax.set_xlabel('相对干预周数（0=干预周）')
    ax.set_ylabel('DID估计的处理效应')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    # 解读
    pre_coefs = [c for p, c in zip(periods, coefficients) if p < 0]
    post_coefs = [c for p, c in zip(periods, coefficients) if p >= 0]
    
    print(f"\n事件研究法解读:")
    print(f"  干预前系数均值: {np.mean(pre_coefs):.3f} (应接近0→平行趋势成立)")
    print(f"  干预后系数均值: {np.mean(post_coefs):.3f} (应为正→处理有效)")
    print(f"  干预后系数趋势: {'上升' if post_coefs[-1] > post_coefs[0] else '下降' if post_coefs[-1] < post_coefs[0] else '平稳'}")
    
    if abs(np.mean(pre_coefs)) < 0.5:
        print(f"  ✓ 干预前系数接近0，平行趋势假设成立")
    else:
        print(f"  ⚠ 干预前系数偏离0，平行趋势可能不成立")
    
    return {'pre_coefs': pre_coefs, 'post_coefs': post_coefs}

event_result = event_study(df_treated, df_control)

### 步骤3：挑战练习——完整评估工具箱

3.1 ITS中断时间序列分析
3.2 安慰剂检验
3.3 效应异质性分析

In [ ]:
# 步骤3.1：ITS（中断时间序列分析）

def its_analysis(df, intervention_week=0):
    """
    ITS模型: Y = β₀ + β₁·Time + β₂·Post + β₃·Time_After
    - β₂: 即刻水平变化（跳变）
    - β₃: 趋势斜率变化（持续效应）
    """
    X = np.column_stack([
        np.ones(n_weeks),  # 截距
        df['week'].values,  # Time: 连续时间变量
        (df['week'] >= intervention_week).astype(int),  # Post: 干预后=1
        np.maximum(df['week'].values - intervention_week + 1, 0)  # Time_After: 干预后时间计数
    ])
    y = df['duration'].values
    
    beta = np.linalg.inv(X.T @ X) @ X.T @ y
    y_pred = X @ beta
    
    # 反事实预测（假设没有干预）
    X_counterfactual = X.copy()
    X_counterfactual[:, 2] = 0  # Post = 0
    X_counterfactual[:, 3] = 0  # Time_After = 0
    y_counterfactual = X_counterfactual @ beta
    
    # 画图
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(df['week'], y, 'b-o', linewidth=1.5, markersize=4, label='实际值')
    ax.plot(df['week'], y_counterfactual, 'r--', linewidth=1.5, alpha=0.7, label='反事实(无干预)')
    ax.axvline(x=-0.5, color='gray', linestyle='--', linewidth=1.5, label='干预时间')
    
    ax.fill_between(df['week'][df['week'] >= 0],
                    y[df['week'] >= 0],
                    y_counterfactual[df['week'] >= 0],
                    alpha=0.2, color='green', label='处理效应')
    
    ax.set_title('ITS: 中断时间序列分析', fontsize=14, fontweight='bold')
    ax.set_xlabel('周（0=干预周）')
    ax.set_ylabel('人均观看时长(分钟)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"\nITS模型结果:")
    print(f"  β₀ (基线): {beta[0]:.2f}")
    print(f"  β₁ (干预前趋势): {beta[1]:.3f}分钟/周")
    print(f"  β₂ (即刻效应/跳变): {beta[2]:+.2f}分钟 ← 干预当周的即时效")
    print(f"  β₃ (趋势变化): {beta[3]:+.3f}分钟/周 ← 干预后每周的额外增长")
    print(f"\n  总效应(干预后12周累计): {beta[2] + beta[3] * 12:+.1f}分钟")
    print(f"  真实效应: {TRUE_EFFECT:.1f}分钟（注意：ITS估计的是'即刻跳变+趋势变化'，和恒定效应不同）")
    
    return beta

its_beta = its_analysis(df)

In [ ]:
# 步骤3.2：安慰剂检验——时间安慰剂

def placebo_test(df, true_intervention=0, n_placebos=100):
    """
    把干预时间随机提前到干预前，看"假干预"能否检测到"效果"
    如果假干预也能检测到效果→说明方法不可靠
    """
    # 只使用干预前数据，在其中随机选一个"假干预时间"
    before_data = df[df['week'] < true_intervention].copy()
    
    placebo_effects = []
    for _ in range(n_placebos):
        # 随机选一个假干预时间（留够前后数据）
        fake_cutoff = np.random.choice(before_data['week'].values[4:-4])
        
        fake_before = before_data[before_data['week'] < fake_cutoff]['duration'].mean()
        fake_after = before_data[before_data['week'] >= fake_cutoff]['duration'].mean()
        placebo_effects.append(fake_after - fake_before)
    
    # 真实效应的朴素估计
    true_before = df[df['week'] < true_intervention]['duration'].mean()
    true_after = df[df['week'] >= true_intervention]['duration'].mean()
    true_naive = true_after - true_before
    
    # 画分布
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(placebo_effects, bins=30, alpha=0.7, color='gray', edgecolor='black', label='安慰剂效应分布')
    ax.axvline(x=true_naive, color='r', linestyle='--', linewidth=2, label=f'真实朴素估计={true_naive:.1f}')
    ax.axvline(x=0, color='black', linestyle='-', linewidth=1)
    
    p_value = np.mean(np.abs(np.array(placebo_effects)) >= np.abs(true_naive))
    
    ax.set_title(f'安慰剂检验 (n={n_placebos})\n真实效应在安慰剂分布中的位置 (p={p_value:.3f})', fontsize=13)
    ax.set_xlabel('估计效应')
    ax.set_ylabel('频数')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
    
    print(f"\n安慰剂检验结果:")
    print(f"  安慰剂效应均值: {np.mean(placebo_effects):.3f}")
    print(f"  安慰剂效应标准差: {np.std(placebo_effects):.3f}")
    print(f"  真实朴素估计: {true_naive:.3f}")
    print(f"  经验p值: {p_value:.3f}")
    
    if p_value < 0.05:
        print(f"  ✓ p<0.05——真实效应在安慰剂分布中是显著异常的→不太可能是随机波动")
    else:
        print(f"  ⚠ p≥0.05——真实效应在安慰剂分布中并不异常→可能只是随机波动")
    
    return placebo_effects, p_value

placebo_effects, p_val = placebo_test(df, true_intervention=0)

In [ ]:
# 步骤3.3：效应异质性——"平均效果"掩盖了什么？

def effect_heterogeneity():
    """
    模拟不同用户群对新功能的反应
    """
    np.random.seed(42)
    n_users = 5000
    
    # 用户特征
    user_groups = np.random.choice(['高活', '中活', '低活', '新用户'], n_users, p=[0.20, 0.35, 0.30, 0.15])
    
    # 不同组对新功能的反应
    effects = {
        '高活': np.random.normal(8.0, 3.0, n_users),   # 高度受益
        '中活': np.random.normal(3.0, 2.5, n_users),   # 中度受益
        '低活': np.random.normal(-1.0, 3.0, n_users),  # 轻微负面
        '新用户': np.random.normal(-5.0, 4.0, n_users)  # 严重负面（功能太复杂）
    }
    
    user_effects = np.array([effects[g][i] for i, g in enumerate(user_groups)])
    
    print("=" * 55)
    print("效应异质性分析")
    print("=" * 55)
    print(f"平均效应: {user_effects.mean():+.2f} → 看起来还不错！")
    print(f"\n但分群后:")
    print(f"{'用户群':<12}{'占比':>8}{'平均效应':>12}{'正效应比例':>12}")
    print("-" * 45)
    
    for group in ['高活', '中活', '低活', '新用户']:
        mask = user_groups == group
        group_effect = user_effects[mask].mean()
        pos_ratio = (user_effects[mask] > 0).mean()
        share = mask.sum() / n_users
        print(f"{group:<12}{share:>7.1%}{group_effect:>+11.2f}{pos_ratio:>11.1%}")
    
    print(f"\n⚠ 关键发现：")
    print(f"  新用户（15%的用户）平均减少了5分钟！")
    print(f"  如果只看平均值(+1.35)，会完全忽略'新用户被伤害'这个事实")
    print(f"  → 需要在分析报告中单独报告'受影响最严重的用户群'")
    
    # 画图
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 各组的分布
    ax1 = axes[0]
    colors = ['green', 'blue', 'orange', 'red']
    for i, (group, color) in enumerate(zip(['高活', '中活', '低活', '新用户'], colors)):
        mask = user_groups == group
        ax1.hist(user_effects[mask], bins=30, alpha=0.5, color=color, label=group)
    ax1.axvline(x=0, color='black', linestyle='-', linewidth=1)
    ax1.axvline(x=user_effects.mean(), color='black', linestyle='--', linewidth=2, label=f'均值={user_effects.mean():.1f}')
    ax1.set_title('各用户群的效应分布', fontsize=13)
    ax1.set_xlabel('处理效应(分钟)')
    ax1.set_ylabel('人数')
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis='y')
    
    # 各组的箱线图
    ax2 = axes[1]
    data_by_group = [user_effects[user_groups == g] for g in ['高活', '中活', '低活', '新用户']]
    bp = ax2.boxplot(data_by_group, labels=['高活', '中活', '低活', '新用户'], patch_artist=True)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.5)
    ax2.axhline(y=0, color='black', linestyle='-', linewidth=1)
    ax2.set_title('效应异质性——箱线图', fontsize=13)
    ax2.set_ylabel('处理效应(分钟)')
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.suptitle('新功能效果不是"一个数"——它因人而异', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()

effect_heterogeneity()

## 📊 区块4：业务场景应用

### 场景1：全量发布新功能，怎么评估效果？

**背景**：产品经理坚持全量发布"智能推荐算法"，上线后发现人均观看时长从80分钟涨到92分钟（+15%）。PM说效果超好，但你知道这不一定全是因为新算法。

**评估方法（五步法）**：

1. **趋势剥离**：用上线前16周的数据拟合趋势线，外推预测"无新功能"下的时长。设趋势为每周+0.5分钟，16周后自然增长=8分钟，扣除后实际效应=12-8=4分钟。

2. **季节性修正**：上线时正好是暑假（6月），历史数据显示暑假期间时长会比平时高约5%。12×0.05=0.6分钟为季节性贡献。扣除后效应=4-0.6=3.4分钟。

3. **新奇效应衰减**：看上线后8周的趋势。如果前2周效应很强（+10分钟），但第3-8周衰减到+2分钟——说明可能有新奇效应。如果衰减不明显（一直稳定+3-4分钟），说明效果是真实的。

4. **相关指标交叉验证**：如果观看时长涨了但"完播率"没变——可能只是用户在"刷"更多短内容，而不是内容质量真的在变好。如果观看时长涨了且用户次日留存也涨了——效果更可信。

5. **结论给置信区间而非点估计**："新推荐算法让观看时长增加了2-6分钟，最可能的估计是3.5分钟。总体置信度约80%——主要不确定性来自我们不能排除季节性因素的干扰。"

### 场景2："应该灰度实验的，但PM已经全量发了"——事后诸葛亮

**背景**：一个新功能已经全量发布2周了，效果很好（+8%转化率）。但你的AB实验报告还没出（需要3周）。PM吐槽："实验太慢了，我们都看到结果了。"

**你的观点（有理有据）**：

> "以下是我对'+8%转化率'的拆解：
>
> 1. 自然趋势：转化率本来每月自然增长约1.5% → 2周约0.75%
> 2. 季节性：现在是促销季，转化率季节性提升约2%
> 3. 新奇效应：新功能第一周通常有5-10%的新奇效应
> 4. 回归均值：上线前几周转化率正好偏低（'下蹲'效应） → 回弹约1-2%
>
> 扣掉这些因素后，真实的处理效应可能是：+8% - 0.75% - 2% - 3%（新奇效应折中）- 1.5% = +0.75%
>
> 所以真实效果可能不到1%，而不是8%。如果当时做了AB实验，我们不需要猜——实验会自动扣除所有非因果因素。"

**启示**：回顾性分析可以给方向，但永远无法替代前瞻性实验。你的职责不是"禁止全量"，而是"在全量时量化不确定性"。

### 场景3："效果在衰减"——从发现到预警

**背景**：新功能上线后前4周效果很好（+12%），但第5-8周效果开始衰减（+12%→+8%→+5%→+3%）。

**分析方法**：
1. 做"时间×效应"的交互分析：将上线后时间分成4个时段，看效应是否呈下降趋势
2. 识别衰减原因：是"新奇效应衰减"还是"用户已经习惯了"还是"功能内容被消耗完"
3. 预测"稳态效应"：用衰减曲线拟合，预测最终稳定在什么水平

**建议**：
- 如果稳态效应仍为正（比如+2-3%）且统计显著→功能值得保留
- 如果衰减趋势指向0或负值→功能可能是个"噱头"，需要计划迭代或回滚
- 建立"效果衰减预警"机制——当衰减速度超过阈值时自动通知产品团队

### 场景4：行业对标——"竞品也做了类似功能"

**背景**：竞品2个月前上线了和你家新功能非常类似的功能。你想知道"我们上线后能不能追赶上竞品的增长"。

**合成控制法应用**：
1. 用竞品作为"对照组"——但竞品也上了新功能，不符合"未受干预"的前提
2. 所以用"竞品上线功能前"的数据来拟合你和竞品的关系
3. 预测：如果你们不上新功能，你们和竞品的差距会怎么变化
4. 你们上线后：实际差距 - 预测差距 = 你们新功能的"追赶效果"

**结论示例**：
> "竞品上线类似功能后，我们的份额下降了约3%。我们上线后，份额在4周内收复了约1.5%。考虑到竞品有2个月的先发优势，我们的功能效果和竞品相当——既没有更好，也没有更差。但要追平竞品，可能需要额外投入（更大的推荐流量入口）。"

### 场景5："负效果"但不敢说的困境

**背景**：分析结果显示新功能实际上降低了用户活跃度（-2%），但产品负责人对这个功能倾注了大量心血，你担心直接汇报负效果会引发抵触。

**策略**（诚实但有技巧的汇报）：
1. 先认可意图："这次改版的出发点很好——让用户更快找到想看的内容"
2. 然后给数据："但从数据上看，改版后用户的平均浏览时长下降了2%（95%CI: -3.5% ~ -0.5%，p=0.01）"
3. 最重要的——给"为什么会这样"和"怎么办"："下钻后发现，主要问题是新布局让老用户找不到熟悉的功能入口（搜索点击率下降了15%）。建议在保留新布局的同时，恢复老用户的'经典模式'选项。"
4. 给"积极的下一步"："我们可以做一个快速迭代——在2周内上线'经典模式开关'，预计可以挽回80%的下降。"

**启示**：报告负效果不是"打脸"，而是"帮助产品更快迭代"。如果你不敢说真话，产品就一直在一个错误的方向上浪费资源。

## ✅ 区块5：分级检验清单

### 🟢 初级检验（必须能独立完成的）
- [ ] 能解释为什么朴素前后对比会高估效果（至少说出3个原因）
- [ ] 能说出DID的基本公式和核心假设
- [ ] 能实现简单的趋势剥离分析
- [ ] 能区分"相关性"和"因果性"

**答案提示**：
- 朴素对比三大高估来源：自然趋势、季节性、新奇效应
- DID = (处理后-处理前) - (对照后-对照前)
- 趋势剥离 = 上线后实际值 - 上线前趋势外推预测值

### 🟡 中级检验（能解释给同事听的原理）
- [ ] 能解释DID的四大假设（平行趋势、共同冲击、无溢出、组合不变）
- [ ] 能设计并执行平行趋势检验
- [ ] 能进行安慰剂检验并解释其意义
- [ ] 能区分ITS模型中"即刻效应"和"趋势变化"

**答案提示**：
- 平行趋势检验：看干预前两组的轨迹是否平行
- 安慰剂检验：把干预时间提前，看是否还能检测到"效果"
- ITS即时效应是跳变，趋势变化是斜率差异

### 🔴 高级检验（能举一反三的迁移能力）
- [ ] 能在没有对照组的情况下设计准实验评估方案
- [ ] 能识别并量化"新奇效应衰减"
- [ ] 能实现效应异质性分析，识别"受益者"和"受害者"
- [ ] 能撰写包含"不确定性量化"的评估报告
- [ ] 能向非技术背景的PM解释"为什么+8%可能不是真的"

**答案提示**：
- 无对照组时用ITS或趋势外推
- 新奇衰减看效应的时间趋势
- CATE（条件平均处理效应）和分组DID
- 报告给置信区间而非点估计
- 用"自然增长+季节性+新奇效应"三因素拆解来向PM解释

## 🔗 区块6：跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **← 前置依赖** | 第3周AB实验 | AB实验是RCT，新功能评估是准实验——原理相通，限制不同 |
| **← 前置依赖** | 第7周因果推断 | DID、ITS、SCM都是因果推断的具体方法 |
| **← 前置依赖** | 第9周统计检验 | 处理效应的显著性检验 |
| **← 前置依赖** | 第4周用户分层 | 效应异质性分析需要用户分层基础 |
| **→ 后续用途** | W11综合项目 | 综合项目中需要评估一个策略的效果 |
| **→ 后续用途** | 战略分析岗位（Senior DA/DS） | 策略评估是核心日常工作 |
| **业务连接** | 产品策略决策 | 新功能效果评估直接决定功能的去留和迭代方向 |
| **业务连接** | 高管汇报 | 需要向非技术高管解释"为什么效果是X而不是Y" |

### 横向连接：与其他分析方法的关联

**与AB实验的连接**：DID和AB实验都是因果推断方法。AB实验是"前瞻性的DID"（通过随机化保证平行趋势），DID是"回顾性的AB实验"（通过假设平行趋势来替代随机化）。

**与时间序列预测的连接**：趋势剥离和ITS本质上都是时间序列模型。Prophet、ARIMA等时间序列预测方法可以替代简单线性趋势进行更准确的"反事实预测"。

**与机器学习的连接**：因果森林（Causal Forest）和元学习器（Meta-learners: S-learner, T-learner, X-learner）是机器学习在因果推断中的前沿应用，可以估计个体层面的处理效应。

**与数据汇报的连接**：新功能评估的结果需要向上汇报。一个好的评估报告要包含：效应大小和置信区间、主要假设和检验结果、效应异质性分析、局限性和下一步建议。

## 📝 区块7：当日小结

> **没有对照组的上线后对比，会把"自然增长"和"季节效应"全部算成策略效果。即使不能做AB实验，至少用趋势外推做一个"准DID"——把自然增长剥离掉。DID的核心是平行趋势假设——如果这个假设不成立，你的整个分析都是建立在一句"假设..."之上的。评估新功能效果的五维框架：直接效果→异质性→溢出效应→持续性→机制验证。不要只报告"平均值"——效应异质性分析可能发现"对A用户有效，对B用户有害"的关键洞察。**

**五条金句带回家**：
1. "上线后涨了X%"≠"功能有效"——你看到的是趋势+季节+噪声+真效果的混合体
2. DID不只是一个公式——它代表一种"用对照组的变化来剥离外部因素"的思维方式
3. 平行趋势假设是DID的生命线——信它之前先检验它
4. 安慰剂检验是因果推断的"谎言检测器"——如果假干预也有"效果"，你的方法就有问题
5. 平均效果可能掩盖极端异质性——"对80%用户有效+5%，对20%用户有害-10%"和"全体有效+2%"是完全不同的故事

### 🔖 课后拓展

1. **练习**：找一个你能获取数据的App，找它最近一次大版本更新的时间，用今天的方法评估这次更新对应用商店评分的影响

2. **思考题**：如果平行趋势假设不成立（干预前两组的趋势已经不同），你还能用什么方法评估处理效应？

3. **预习**：明天的案例——渠道预算分配中的归因模型和边际ROI分析

4. **面试准备**：背诵以下问题的回答思路：
   - "PM说上线后指标涨了10%，你怎么判断是不是真的有效？"
   - "如果没有AB实验，你怎么评估策略效果？"
   - "DID的平行趋势假设是什么意思？怎么检验？"

---

*03analyst_thinking · Week 10 Day 3 · 2026-05-21*